In [1]:
!pip install cirq qsimcirq brain2
import numpy as np
import cirq
import qsimcirq

from brian2 import *
import brian2cuda

# -------------------------
# 1) Quantum stage (Cirq + qsimcirq)
# -------------------------
n = 4
qs = cirq.LineQubit.range(n)
circuit = cirq.Circuit(
    cirq.H.on_each(*qs),
    cirq.CNOT(qs[0], qs[1]),
    cirq.CNOT(qs[1], qs[2]),
    cirq.CNOT(qs[2], qs[3]),
    cirq.measure(*qs, key="m")
)

# qsimcirq simulator (GPU if built/installed with GPU support)
sim = qsimcirq.QSimSimulator()

# Sample bitstrings
reps = 2000
result = sim.run(circuit, repetitions=reps)
bits = result.measurements["m"]  # shape: (reps, n), dtype=0/1

# Convert quantum samples -> a vector in [0, 1]
p1 = bits.mean(axis=0)  # probability of measuring 1 on each qubit
# Example mapping: turn probabilities into input currents
Ivals = 0.5 + 2.0 * p1  # arbitrary mapping (n values)

# -------------------------
# 2) Brian2 stage (CUDA via brian2cuda)
# -------------------------
set_device("cuda_standalone", build_on_run=False)

start_scope()

N = n
tau = 10*ms
eqs = """
dv/dt = (-v + I)/tau : 1
I : 1
"""

G = NeuronGroup(N, eqs, threshold="v>1.0", reset="v=0", method="euler")
G.I = Ivals  # inject quantum-derived current
G.v = 0

# Simple recurrent synapses with quantum-derived weights
# (here: outer product just for demo)
W = np.outer(p1, p1)
S = Synapses(G, G, "w : 1", on_pre="v_post += w")
S.connect(condition="i!=j")
S.w = W[S.i, S.j]

spk = SpikeMonitor(G)
state = StateMonitor(G, "v", record=True)

run(200*ms)

device.build(directory="brian2cuda_build", compile=True, run=True)

# -------------------------
# 3) "Program output" is Brian2
# -------------------------
print("Spike counts:", spk.count[:])
print("First spikes (neuron, time):", list(zip(spk.i[:10], spk.t[:10])))

ERROR: Ignored the following versions that require a different python version: 0.3.1.27 Requires-Python ==2.7.*; 0.5.555 Requires-Python >=2.7,<=3.5; 0.5.556 Requires-Python ==3.5.*
ERROR: Could not find a version that satisfies the requirement brain2 (from versions: none)
ERROR: No matching distribution found for brain2


ModuleNotFoundError: No module named 'cirq'